## Ein Pydantic-Model erstellen

Wie lässt sich festlegen, welche Daten ein Objekt haben muss, und wie wird das beim Erzeugen automatisch geprüft? Genau dafür dient ein Model: Es beschreibt die Daten (Namen und Typen der Fields), Pydantic übernimmt die Validierung.

In diesem Notebook definieren wir ein `Person`-Model, sehen uns an, was Pydantic daraus macht, und prüfen, wann Fehler auftreten und wann nicht.

**Danach ist klar:**
- wie ein Model mit `BaseModel` definiert wird,
- was Pydantic beim Erzeugen prüft und umwandelt,
- wann Fehler auftreten und wann nicht.

Voraussetzung: Grundkenntnisse zu Klassen und Type Hints in Python.

In [ ]:
# `BaseModel` ist die Basisklasse aller Pydantic-Models.
from pydantic import BaseModel

In [ ]:
# Die Annotationen legen Fields und Typen fest, Pydantic baut daraus die Validierung. Ohne Default-Werte sind alle drei Fields Pflicht.
class Person(BaseModel):
    first_name: str
    last_name: str
    age: int

In [ ]:
# Models akzeptieren nur Keyword-Argumente. Pydantic prüft die Werte gegen die Annotationen, bevor die Instanz entsteht.
p = Person(first_name="Evariste", last_name="Galois", age=20)
p

In [ ]:
# `str(p)` liefert die lesbare Darstellung mit Feldnamen und Werten, aber ohne Klassennamen.
str(p)

In [ ]:
# `repr(p)` ergänzt den Klassennamen und sieht aus wie der Aufruf, der die Instanz erzeugt.
repr(p)

Welche Daten ein Model beschreibt, steht nicht in der Instanz, sondern in der Klasse. Wie lässt sich das einsehen?

In [ ]:
# `model_fields` enthält die Metadaten aller Fields (Annotation, `required=True`). Sie gehören zur Klasse, deshalb steht hier `type(p)`;
# der Zugriff über die Instanz ist seit Pydantic 2.11 veraltet.
type(p).model_fields

In [ ]:
# Dasselbe Ergebnis, diesmal direkt über den Klassennamen.
Person.model_fields

### Validierungsfehler

Was passiert, wenn Daten nicht zum Model passen? Pydantic löst dann eine `ValidationError` aus, statt ungültige Daten stillschweigend zu übernehmen.

In [ ]:
# `ValidationError` ist die Exception, die Pydantic bei ungültigen Daten auslöst.
from pydantic import ValidationError

**Vorhersage:** Im folgenden Aufruf fehlen zwei der drei Fields. Meldet Pydantic nur den ersten Fehler oder alle?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Der Aufruf lässt `first_name` und `age` weg. Pydantic sammelt alle Fehler und meldet sie gemeinsam in einer `ValidationError`. Jeder Eintrag nennt Feld, Fehlertyp (`missing`) und Input.

Kurz gesagt: Ein Aufruf liefert alle Probleme auf einmal.
</details>

In [ ]:
try:
    Person(last_name="Galois")
except ValidationError as exc:
    print(exc)

### Abgeleitete Werte mit `@property`

Wie lässt sich ein Wert anbieten, der sich aus den Fields ableiten lässt, ohne ihn zusätzlich zu speichern? Eine `@property` berechnet ihn bei jedem Zugriff neu.

In [ ]:
# `display_name` ist kein Field, sondern wird bei jedem Zugriff aus `first_name` und `last_name` berechnet und kann deshalb nicht abweichen.
# Die Property erscheint weder in `model_fields` noch in `model_dump()`; dafür gibt es `@computed_field`.
class Person(BaseModel):
    first_name: str
    last_name: str
    age: int

    @property
    def display_name(self):
        return f"{self.first_name} {self.last_name}"

In [ ]:
p = Person(first_name="Evariste", last_name="Galois", age=20)

In [ ]:
# `id(p)` ist die Speicheradresse der Instanz, `hex` stellt sie lesbar dar. Sie dient gleich als Vergleichswert.
hex(id(p))

In [ ]:
# Die Property wird ohne Klammern aufgerufen und setzt den Wert aus den aktuellen Fields zusammen.
p.display_name

In [ ]:
# Fields lassen sich wie Attribute lesen.
p.first_name

In [ ]:
# Der Wert ist ein `int`, wie annotiert.
p.age

### Zuweisung und Validierung

Sind Models nach dem Erzeugen unveränderlich, und wird bei einer späteren Zuweisung erneut geprüft?

In [ ]:
# `age` wird direkt an der Instanz geändert, denn Models sind standardmäßig veränderbar.
p.age = 21

In [ ]:
# `age=21` zeigt: Die bestehende Instanz wurde verändert, es gibt keine Kopie.
p

In [ ]:
# Dieselbe Adresse wie vorher: Die Zuweisung hat die Instanz verändert, nicht ersetzt.
hex(id(p))

In [ ]:
# `"twenty"` lässt sich nicht in `int` umwandeln (Fehlertyp `int_parsing`), `"20"` wäre zu `20` konvertiert worden.
# Der Strict Mode würde die Konvertierung ganz abschalten.
try:
    Person(first_name="Evariste", last_name="Galois", age="twenty")
except ValidationError as exc:
    print(exc)

In [ ]:
# `p` ist unverändert, der fehlgeschlagene Aufruf hat die bestehende Instanz nicht berührt.
p

In [ ]:
# Eine gültige Zuweisung setzt `age` wieder auf `20`.
p.age = 20

**Vorhersage:** Jetzt wird `"twenty"` einem bestehenden Objekt zugewiesen. Löst das einen Fehler aus? Und was steht danach in `p`?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Keine Exception: Pydantic validiert standardmäßig nur beim Erzeugen. Bei Zuweisungen muss das mit `ConfigDict(validate_assignment=True)` eingeschaltet werden.
</details>

In [ ]:
p.age = "twenty"

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `p` zeigt `age='twenty'`, einen String im `int`-Field.

Kurz gesagt: Gültige Daten garantiert Pydantic beim Erzeugen, bei Zuweisungen nur mit `validate_assignment=True`.
</details>

In [ ]:
p

### Praxisbeispiel: Datensätze einlesen

Daten aus einer API oder Datei liegen meist als Dictionaries vor, oft mit kleinen Unsauberkeiten. Wie verarbeitet `Person` solche Einträge, und wie lassen sich fehlerhafte aussortieren?

**Vorhersage:** Der zweite Eintrag enthält das Alter als String `"41"`, der dritte hat keinen `last_name`. Welche Einträge werden akzeptiert, und welchen Typ hat `age` danach?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Die Schleife erzeugt je Eintrag eine `Person`. `**entry` entpackt das Dictionary in Keyword-Argumente. Ungültige Einträge fängt `except ValidationError` ab, die Schleife läuft weiter. `exc.error_count()` liefert die Zahl der Fehler.

Der String `"41"` wird zu `41` umgewandelt (Coercion). Beim dritten Eintrag fehlt `last_name`.

Kurz gesagt: Das Model nimmt kleine Unsauberkeiten an und lehnt Unvollständiges ab.
</details>

In [ ]:
entries = [
    {"first_name": "Ada", "last_name": "Lovelace", "age": 36},
    {"first_name": "Alan", "last_name": "Turing", "age": "41"},
    {"first_name": "Grace", "age": 85},
]

for entry in entries:
    try:
        person = Person(**entry)
    except ValidationError as exc:
        print(f"Ungültiger Eintrag {entry}: {exc.error_count()} Fehler")
    else:
        print(f"{person.display_name}, {person.age} Jahre (Typ: {type(person.age).__name__})")

### Zusammenfassung

- `BaseModel` plus Annotationen definiert ein Model; alle Fields ohne Default sind Pflicht.
- `Person(...)` validiert beim Erzeugen, wandelt passende Werte um (`"41"` → `41`) und meldet alle Fehler gesammelt in einer `ValidationError`.
- `model_fields` gehört zur Klasse und wird über `Person.model_fields` oder `type(p).model_fields` gelesen.
- Eine `@property` liefert abgeleitete Werte, erscheint aber nicht in `model_fields` und `model_dump()`.
- Validiert wird nur beim Erzeugen der Instanz: Spätere Zuweisungen (`p.age = "abc"`) werden standardmäßig nicht geprüft; dafür dient `validate_assignment=True`.